# 🎯 Answer Accuracy Evaluation — Demo Notebook

A guided workflow for evaluating whether a model can answer factual questions correctly across domains such as history, science, math, law, medicine, finance, literature, and business-specific knowledge.

This notebook is intentionally **demo-style and user-friendly**: the reusable implementation lives in `src/genai_capability_bench`, while the notebook focuses on configuration, execution, diagnostics, visualization, and reporting.

---

## What This Tests

| Question | How this notebook evaluates it |
|---|---|
| Did the model answer correctly? | Reference-answer matching and semantic similarity |
| Where is the model strongest or weakest? | Domain/category slicing |
| Which cases need human review? | Low-score and near-threshold diagnostics |
| Is the run reproducible? | Saved config, raw outputs, CSV/JSON artifacts, and summary report |

**Current starter metrics:** exact match, contains match, token F1, and local TF-IDF similarity. Later versions can plug in embedding similarity and LLM-as-judge grading without changing this notebook flow.

---

## 📚 Dataset Options for Answer Accuracy

The built-in sample dataset is intentionally tiny so this notebook can run without credentials. For more serious benchmarking, consider these datasets or dataset families:

| Dataset / Source | Best for | Notes |
|---|---|---|
| **MMLU** | Broad subject coverage | 57 tasks across STEM, humanities, law, medicine, business, and social sciences |
| **TriviaQA** | Open-domain factual QA | Good for general knowledge and retrieval-optional QA |
| **Natural Questions** | Real user-style QA | Useful for practical search-like factual questions |
| **SQuAD** | Reading comprehension | Better for context-grounded QA than pure closed-book accuracy |
| **ARC** | Science exam QA | Useful for grade-school science and reasoning-heavy science questions |
| **HotpotQA** | Multi-hop QA | Better handled later in the RAG/reasoning notebooks, but useful for multi-step answer accuracy |
| **TruthfulQA** | Misconception-resistant answers | Covered more directly in the Truthfulness notebook |
| **Custom golden sets** | Enterprise/domain QA | Recommended for production evaluation: internal policies, product docs, legal/compliance Q&A, support knowledge bases |

> Practical tip: start with a small curated golden set before running large public benchmarks. A high-quality 100-question domain set is often more useful than a noisy 10,000-question benchmark.

---

## ⚙️ Configuration

Choose the dataset, sample size, category slice, model config, and pass threshold before running the evaluation.

The notebook supports two configuration styles:

1. **Manual config variables** — always works in Jupyter, VS Code, Colab, and Codex notebooks.
2. **Optional widgets** — set `USE_INTERACTIVE_WIDGETS = True` if your notebook frontend renders `ipywidgets` correctly. If you see raw text like `VBox(children=...)`, set it back to `False` and use the manual variables.

| Setting | Meaning |
|---|---|
| `DATASET_KEY` | Which registered dataset to evaluate. |
| `CUSTOM_DATASET_PATH` | Optional path for your own JSON/JSONL/CSV task file. |
| `SAMPLE_SIZE` | Number of answer-accuracy questions to run. Use a small number for smoke tests. |
| `SELECTED_CATEGORIES` | Domain slices to include. Use `"ALL"` for broad coverage. |
| `PASS_THRESHOLD` | Minimum score required to count a response as passed. |
| `CONFIG_PATH` | Model/provider config. Use mock config for local workflow tests or real-model template for `.env`-based runs. |

For real model runs, use `configs/eval_openai_compatible_template.yaml` and make sure `.env` contains your provider credentials and deployment/model names.


In [ ]:
# =============================================================================
# CELL 1: IMPORTS & NOTEBOOK SETUP
# =============================================================================
from pathlib import Path
from datetime import datetime
import json
import sys
import warnings

warnings.filterwarnings('ignore')

# Keep the notebook runnable from a fresh checkout without requiring installation.
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
    WIDGETS_IMPORTABLE = True
except Exception:
    WIDGETS_IMPORTABLE = False

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

from genai_capability_bench.capabilities.registry import get_evaluator
from genai_capability_bench.clients.factory import create_client
from genai_capability_bench.core.runner import load_config, parse_models
from genai_capability_bench.core.schemas import Capability
from genai_capability_bench.datasets import materialize_dataset
from genai_capability_bench.datasets.registry import dataset_options_table
from genai_capability_bench.reporting.executive_summary import generate_summary
from genai_capability_bench.reporting.plots import plot_capability_scores
from genai_capability_bench.reporting.tables import capability_leaderboard, summarize_results

CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_core_demo.yaml'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs'

print('✅ Imports loaded')
print(f'📁 Repo root: {REPO_ROOT}')
print(f'⚙️ Config:    {CONFIG_PATH}')

In [ ]:
# =============================================================================
# CELL 2: CONFIGURATION BLOCK — edit these values before running
# =============================================================================
# Optional: set to True only if your notebook frontend renders ipywidgets correctly.
# If you see raw text like "VBox(children=...)", keep this False.
USE_INTERACTIVE_WIDGETS = False

# Choose the model/provider config.
# - eval_core_demo.yaml: mock model, no credentials, safe smoke test
# - eval_openai_compatible_template.yaml: real target LLM using .env
CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_core_demo.yaml'
# CONFIG_PATH = REPO_ROOT / 'configs' / 'eval_openai_compatible_template.yaml'

# ---- Dataset selection -----------------------------------------------------
# Public datasets will be downloaded from Hugging Face if not cached locally.
# Local/custom datasets do not require internet.
DATASET_KEY = 'answer_accuracy_sample'
# Options include:
#   answer_accuracy_sample, core_demo_mixed, mmlu, triviaqa, natural_questions,
#   squad, arc, hotpotqa, truthfulqa, custom
CUSTOM_DATASET_PATH = None  # example: REPO_ROOT / 'datasets' / 'my_golden_set.json'
DATASET_SPLIT = None        # None = dataset default split; examples: validation, test, train
DOWNLOAD_IF_MISSING = True
CACHE_LOCAL_COPY = True
REFRESH_DATASET_CACHE = False

# ---- Evaluation settings ---------------------------------------------------
SAMPLE_SIZE = 10            # use a small number for first run; public datasets can be large
RANDOM_SEED = 42
SELECTED_CATEGORIES = 'ALL' # use 'ALL' or a list like ['history', 'science']
PASS_THRESHOLD = None       # None = read from config default_pass_threshold
RUN_ID_PREFIX = 'answer_accuracy_demo'
# ---------------------------------------------------------------------------

config = load_config(CONFIG_PATH)

print('📚 Available registered datasets')
display(dataset_options_table())

# Optional widgets: useful in classic Jupyter when ipywidgets renders correctly.
if USE_INTERACTIVE_WIDGETS and WIDGETS_IMPORTABLE:
    dataset_options = dataset_options_table()['key'].tolist()
    dataset_widget = widgets.Dropdown(
        options=dataset_options,
        value=DATASET_KEY,
        description='Dataset',
        style={'description_width': 'initial'},
    )
    sample_size_widget = widgets.IntText(value=SAMPLE_SIZE, description='Sample size', style={'description_width': 'initial'})
    threshold_widget = widgets.FloatText(
        value=float(config.get('default_pass_threshold', 0.7)) if PASS_THRESHOLD is None else float(PASS_THRESHOLD),
        description='Pass threshold',
        style={'description_width': 'initial'},
    )
    run_prefix_widget = widgets.Text(value=RUN_ID_PREFIX, description='Run ID prefix', style={'description_width': 'initial'})
    display(widgets.VBox([dataset_widget, sample_size_widget, threshold_widget, run_prefix_widget]))
    print('Widget mode enabled. After changing widget values, run the next cells.')
elif USE_INTERACTIVE_WIDGETS and not WIDGETS_IMPORTABLE:
    print('ipywidgets is not installed. Falling back to manual config variables.')

if USE_INTERACTIVE_WIDGETS and WIDGETS_IMPORTABLE:
    DATASET_KEY = dataset_widget.value
    SAMPLE_SIZE = int(sample_size_widget.value)
    PASS_THRESHOLD = float(threshold_widget.value)
    RUN_ID_PREFIX = run_prefix_widget.value.strip() or 'answer_accuracy_demo'

all_tasks, dataset_path = materialize_dataset(
    DATASET_KEY,
    repo_root=REPO_ROOT,
    split=DATASET_SPLIT,
    sample_size=SAMPLE_SIZE,
    seed=RANDOM_SEED,
    download_if_missing=DOWNLOAD_IF_MISSING,
    cache_local_copy=CACHE_LOCAL_COPY,
    refresh_cache=REFRESH_DATASET_CACHE,
    custom_path=CUSTOM_DATASET_PATH,
)
accuracy_tasks_all = [t for t in all_tasks if t.capability == Capability.ANSWER_ACCURACY]
available_categories = sorted({t.category for t in accuracy_tasks_all})

if SELECTED_CATEGORIES == 'ALL':
    SELECTED_CATEGORIES = available_categories

if PASS_THRESHOLD is None:
    PASS_THRESHOLD = float(config.get('default_pass_threshold', 0.7))

settings_df = pd.DataFrame([
    {'Setting': 'Model config', 'Value': str(CONFIG_PATH.relative_to(REPO_ROOT))},
    {'Setting': 'Dataset key', 'Value': DATASET_KEY},
    {'Setting': 'Dataset source/cache', 'Value': str(dataset_path.relative_to(REPO_ROOT)) if dataset_path and dataset_path.is_relative_to(REPO_ROOT) else str(dataset_path)},
    {'Setting': 'Dataset split', 'Value': DATASET_SPLIT or 'default'},
    {'Setting': 'Download if missing', 'Value': DOWNLOAD_IF_MISSING},
    {'Setting': 'Cache local copy', 'Value': CACHE_LOCAL_COPY},
    {'Setting': 'Available categories', 'Value': ', '.join(available_categories)},
    {'Setting': 'Selected categories', 'Value': ', '.join(SELECTED_CATEGORIES)},
    {'Setting': 'Sample size', 'Value': SAMPLE_SIZE},
    {'Setting': 'Pass threshold', 'Value': PASS_THRESHOLD},
    {'Setting': 'Run ID prefix', 'Value': RUN_ID_PREFIX},
    {'Setting': 'Configured models', 'Value': ', '.join(m['name'] for m in config.get('models', []))},
])

print('✅ Configuration loaded')
print('Tip: edit DATASET_KEY, SAMPLE_SIZE, SELECTED_CATEGORIES, CONFIG_PATH, or DATASET_SPLIT in this cell, then re-run cells below.')
display(settings_df)


---

## 📦 Import / Preparation

This section confirms the config, dataset path, model definitions, and output directory. A good notebook should fail early if a path or config is wrong.

In [ ]:
# =============================================================================
# CELL 3: PREPARATION CHECKS
# =============================================================================
# dataset_path is selected in the configuration cell.
models = parse_models(config)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

prep_rows = [
    {'Check': 'Config file exists', 'Status': CONFIG_PATH.exists(), 'Detail': str(CONFIG_PATH)},
    {'Check': 'Dataset source/cache exists', 'Status': dataset_path is None or dataset_path.exists(), 'Detail': str(dataset_path)},
    {'Check': 'Models configured', 'Status': len(models) > 0, 'Detail': ', '.join(m.name for m in models)},
    {'Check': 'Output directory ready', 'Status': OUTPUT_ROOT.exists(), 'Detail': str(OUTPUT_ROOT)},
]
prep_df = pd.DataFrame(prep_rows)
display(prep_df)

if not all(prep_df['Status']):
    raise RuntimeError('❌ Preparation failed. Fix the failed checks before continuing.')

print('✅ Preparation complete')

---

## 📂 Data Loading

We load all tasks from the configured dataset, then filter to `answer_accuracy`. The preview helps confirm that categories, references, and expected answers look sensible before spending tokens on a run.

In [ ]:
# =============================================================================
# CELL 4: LOAD AND FILTER DATA
# =============================================================================
accuracy_tasks = [t for t in accuracy_tasks_all if t.category in SELECTED_CATEGORIES]
accuracy_tasks = accuracy_tasks[:SAMPLE_SIZE]

task_preview = pd.DataFrame([
    {
        'task_id': t.task_id,
        'category': t.category,
        'question': t.input_text,
        'expected_output': t.expected_output,
        'references': '; '.join(t.references),
    }
    for t in accuracy_tasks
])

print(f'✅ Loaded {len(all_tasks)} total tasks')
print(f'🎯 Selected {len(accuracy_tasks)} answer-accuracy tasks')
print(f'🏷️ Categories: {SELECTED_CATEGORIES}')
display(task_preview)


---

## ✅ Pre-Flight Check

Before running the evaluation, verify that every selected task has the fields needed for answer accuracy scoring.

In [ ]:
# =============================================================================
# CELL 5: PRE-FLIGHT CHECK
# =============================================================================
checks = []
for task in accuracy_tasks:
    checks.append({
        'task_id': task.task_id,
        'has_question': bool(task.input_text),
        'has_expected_or_reference': bool(task.expected_output or task.references),
        'category': task.category,
    })

check_df = pd.DataFrame(checks)
display(check_df)

if check_df.empty:
    raise RuntimeError('❌ No answer-accuracy tasks selected.')
if not check_df[['has_question', 'has_expected_or_reference']].all().all():
    raise RuntimeError('❌ Some tasks are missing required fields.')

print('✅ Pre-flight passed. Ready to evaluate.')

---

## 🤖 Testing / Evaluation Run

This cell evaluates each selected model against each selected answer-accuracy task. It shows a progress bar and saves raw results to `outputs/runs/<run_id>/`.

For long runs, start small first. Once the smoke test passes, increase sample size or switch to a larger dataset.

In [ ]:
# =============================================================================
# CELL 6: RUN EVALUATION WITH PROGRESS BAR
# =============================================================================
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_id = f'{RUN_ID_PREFIX}_{timestamp}'
run_dir = OUTPUT_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=True)

print(f'🚀 Starting run: {run_id}')
print(f'📊 Models: {[m.name for m in models]}')
print(f'🧪 Tasks:  {len(accuracy_tasks)}')

results = []
for model in models:
    print(f'\n🔌 Initializing client: {model.name} ({model.provider})')
    client = create_client(model)
    evaluator = get_evaluator(Capability.ANSWER_ACCURACY, pass_threshold=PASS_THRESHOLD)
    for task in tqdm(accuracy_tasks, desc=f'Evaluating {model.name}'):
        result = evaluator.evaluate_task(run_id, task, model, client)
        results.append(result.to_dict())

results_df = pd.DataFrame(results)
summary_df = summarize_results(results)
leaderboard_df = capability_leaderboard(results)

(run_dir / 'results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
results_df.to_csv(run_dir / 'results.csv', index=False)
summary_df.to_csv(run_dir / 'summary.csv', index=False)
leaderboard_df.to_csv(run_dir / 'leaderboard.csv', index=False)

print('\n✅ Evaluation complete')
print(f'📁 Artifacts saved to: {run_dir}')
display(results_df[['model_name', 'task_id', 'category', 'actual_output', 'expected_output', 'score', 'passed']])

---

## 🔎 Result Analysis and Diagnostics

This section summarizes performance by category and highlights cases that deserve review. In a real evaluation, the most valuable examples are often not the average cases; they are the low-score, near-threshold, or surprising failures.

In [ ]:
# =============================================================================
# CELL 7: SUMMARY TABLES AND DIAGNOSTICS
# =============================================================================
print('📋 Summary by model / capability / category')
display(summary_df)

print('🏁 Capability leaderboard')
display(leaderboard_df)

diagnostic_df = results_df.copy()
diagnostic_df['review_priority'] = pd.cut(
    diagnostic_df['score'],
    bins=[-0.01, 0.4, PASS_THRESHOLD, 1.0],
    labels=['High review priority', 'Near threshold', 'Looks good'],
)

print('🩺 Diagnostic cases')
display(
    diagnostic_df.sort_values(['score', 'task_id'])[
        ['review_priority', 'model_name', 'task_id', 'category', 'actual_output', 'expected_output', 'score', 'passed']
    ]
)

---

## 📊 Visualization

The first chart shows average score by category. The second chart shows the score distribution across selected tasks. With larger datasets, these charts make weak domains and unstable performance easier to spot.

In [ ]:
# =============================================================================
# CELL 8: VISUALIZATION
# =============================================================================
plot_capability_scores(summary_df, title='Answer Accuracy by Category');

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(results_df['score'], bins=10, color='#2563eb', alpha=0.85, edgecolor='white')
ax.axvline(PASS_THRESHOLD, color='#dc2626', linestyle='--', label=f'Pass threshold ({PASS_THRESHOLD:.2f})')
ax.set_title('Answer Accuracy Score Distribution')
ax.set_xlabel('Score')
ax.set_ylabel('Number of responses')
ax.set_xlim(0, 1)
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

---

## 📝 Executive Summary / Reporting

This section generates a concise, deterministic summary and writes a Markdown report to the run folder. For governed workflows, this report can be attached to model comparison records, validation evidence, or regression-test artifacts.

In [ ]:
# =============================================================================
# CELL 9: EXECUTIVE SUMMARY AND REPORT
# =============================================================================
summary_text = generate_summary(summary_df)

report = f'''# Answer Accuracy Evaluation Report

**Run ID:** `{run_id}`  
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Models:** {', '.join(m.name for m in models)}  
**Tasks evaluated:** {len(accuracy_tasks)}  
**Pass threshold:** {PASS_THRESHOLD:.2f}

## Executive Summary

{summary_text}

## Artifacts

- `results.json`: raw normalized capability results
- `results.csv`: tabular result details
- `summary.csv`: category-level score summary
- `leaderboard.csv`: model/capability leaderboard

## Recommended Review

Review all high-priority and near-threshold cases before using these results for model selection or governance evidence.
'''

(run_dir / 'answer_accuracy_report.md').write_text(report, encoding='utf-8')

display(Markdown('### Executive Summary'))
display(Markdown(summary_text))
print(f'📄 Report saved to: {run_dir / "answer_accuracy_report.md"}')

---

## ✅ Next Steps

1. Replace the tiny sample dataset with a larger answer-accuracy benchmark or a custom golden set.
2. Add more categories so the scorecard becomes a real capability profile.
3. Switch from the mock model to an OpenAI-compatible or Azure OpenAI provider config.
4. Add embedding-based semantic similarity for stronger open-ended answer scoring.
5. Compare multiple models and use `05_core_llm_leaderboard.ipynb` to aggregate results.

This notebook should serve as the style template for the remaining core capability notebooks.